# Fase 1 - Modelo Predictivo NYC Taxi Trip Duration

En este notebook se entrena un modelo de regresión para predecir la duración de viajes de taxi en Nueva York usando un flujo simple de Machine Learning.

In [4]:
import sys
!{sys.executable} -m pip install scikit-learn joblib -q


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1) Importar librerías

Este bloque importa las librerías necesarias para cargar datos, procesarlos, entrenar el modelo, evaluarlo y guardarlo.

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import joblib
from pathlib import Path

## 2) Cargar el dataset y preprocesar datos

Aquí se carga el archivo `../data/train.csv`, se seleccionan únicamente las columnas requeridas y se eliminan los valores nulos.

In [6]:
columns = [
    "passenger_count",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "trip_duration",
]

df = pd.read_csv("../data/train.csv")
df = df[columns].dropna().copy()

print(f"Filas después de limpiar nulos: {len(df)}")
df.head()

Filas después de limpiar nulos: 1458644


,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,trip_duration
0,1,-73.982155,40.767937,-73.964630,40.765602,455
1,1,-73.980415,40.738564,-73.999481,40.731152,663
2,1,-73.979027,40.763939,-74.005333,40.710087,2124
3,1,-74.010040,40.719971,-74.012268,40.706718,429
4,1,-73.973053,40.793209,-73.972923,40.782520,435


## 3) Crear la feature `trip_distance` con fórmula Haversine

En este bloque se calcula la distancia aproximada entre punto de recogida y punto de destino en kilómetros.

In [7]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """Calcula la distancia Haversine en kilómetros entre dos puntos geográficos."""
    r = 6371.0  # radio aproximado de la Tierra en km

    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)

    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad

    a = np.sin(dlat / 2) ** 2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))

    return r * c


df["trip_distance"] = haversine_distance(
    df["pickup_latitude"],
    df["pickup_longitude"],
    df["dropoff_latitude"],
    df["dropoff_longitude"],
)

df[["trip_distance", "trip_duration"]].head()

,trip_distance,trip_duration
0,1.498521,455
1,1.805507,663
2,6.385098,2124
3,1.485498,429
4,1.188588,435


## 4) Definir `X` e `y` y dividir datos (80/20)

Se separa la variable objetivo `trip_duration` y se realiza una división entrenamiento/prueba para evaluar el modelo.

In [8]:
feature_cols = [
    "passenger_count",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "trip_distance",
]

X = df[feature_cols]
y = df["trip_duration"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Tamaño de entrenamiento: {X_train.shape}")
print(f"Tamaño de prueba: {X_test.shape}")

Tamaño de entrenamiento: (1166915, 6)
Tamaño de prueba: (291729, 6)


## 5) Entrenar el modelo RandomForestRegressor

Se entrena un modelo de bosque aleatorio con parámetros simples.

In [1]:
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

print("Modelo entrenado correctamente.")

NameError: name 'RandomForestRegressor' is not defined

## 6) Evaluar el modelo con RMSE

Se generan predicciones sobre el conjunto de prueba y se calcula la raíz del error cuadrático medio (RMSE).

In [2]:
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"RMSE del modelo: {rmse:,.2f}")

NameError: name 'model' is not defined

## 7) Guardar el modelo entrenado

Se guarda el modelo en la ruta `../models/model.pkl` para su uso posterior.

In [ ]:
model_path = Path("../models/model.pkl")
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model, model_path)

print(f"Modelo guardado en: {model_path.resolve()}")

## 8) Probar el modelo con algunas filas de prueba

Se muestran predicciones del modelo junto con el valor real para una revisión rápida.

In [ ]:
sample_size = 5
X_sample = X_test.head(sample_size)
y_real = y_test.head(sample_size).reset_index(drop=True)
y_sample_pred = model.predict(X_sample)

resultados = pd.DataFrame({
    "real": y_real,
    "prediccion": y_sample_pred,
})

resultados